1️⃣ PromptTemplate：最基础的文本模板

场景：单轮任务 / 传统 completion 模型 / 直接拼字符串

In [ ]:
from langchain.prompts import PromptTemplate

# 1. 定义模板（from_template 会自动推断 input_variables）
summary_prompt = PromptTemplate.from_template(
    "请用一句话总结以下内容：{text}"
)

# 2. 只当字符串模板使用
text = "YOLOv8 在火焰与烟雾检测任务上达到了 mAP50 0.93。"
final_prompt = summary_prompt.format(text=text)
print(final_prompt)
# -> "请用一句话总结以下内容：YOLOv8 在火焰与烟雾检测任务上达到了 mAP50 0.93。"

✅ 技巧：多个变量 + 默认参数封装

In [ ]:
from langchain.prompts import PromptTemplate

qa_prompt = PromptTemplate.from_template(
    "请用{style}的语气回答下面的问题：\n"
    "问题：{question}\n"
    "上下文：{context}\n"
)

def build_prompt(question, context, style="严谨专业"):
    return qa_prompt.format(question=question, context=context, style=style)

print(build_prompt("什么是 Milvus？", "Milvus 是一个向量数据库..."))

统一说明：
	•	用的是 langchain_openai.ChatOpenAI，你可以把 base_url / api_key / model 改成 DashScope 或 SiliconFlow。
	•	为了简单，统一用 StrOutputParser 把结果转成字符串。

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
llm = ChatOpenAI(
    api_key="sk-ruourlqqajhathtbsgmywvanbywoikkwkyhczzqemhkrxcuu",
    base_url="https://api.siliconflow.cn/v1",  # 注意是 base_url
    model="Qwen/Qwen2.5-VL-72B-Instruct",  # 和你 payload 里的 model 保持一致
    temperature=0.7,
    streaming=False,
)
parser = StrOutputParser()

# 1️⃣ PromptTemplate + LLM

In [ ]:
from langchain.prompts import PromptTemplate

summary_prompt = PromptTemplate.from_template(
    "请用一句话总结以下内容：{text}"
)

summary_chain = summary_prompt | llm | parser

text = "YOLOv8 在火焰与烟雾检测任务上达到了 mAP50 0.93，可用于实时监控。"
resp = summary_chain.invoke({"text": text})

print("【PromptTemplate 总结】")
print(resp)

# 2️⃣ ChatPromptTemplate（字符串版）+ LLM

In [ ]:
from langchain.prompts.chat import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template("""
你是一个中文写作助手，请将下面的内容改写得更口语化但保持原意不变：
内容：{text}
""")

chat_chain = chat_prompt | llm | parser

resp = chat_chain.invoke({
    "text": "本系统主要用于对工业园区的火焰与烟雾进行实时检测和告警。"
})

print("【ChatPromptTemplate 改写】")
print(resp)

# 3️⃣ ChatPromptTemplate（多消息版：system + human）+ LLM

In [ ]:
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

system_prompt = SystemMessagePromptTemplate.from_template(
    "你是一个资深 CV 工程师，请用通俗语言回答问题。"
)
human_prompt = HumanMessagePromptTemplate.from_template(
    "请解释一下什么是 YOLO，并举一个实际应用场景。"
)

multi_chat_prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    human_prompt,
])

multi_chat_chain = multi_chat_prompt | llm | parser

resp = multi_chat_chain.invoke({})
print("【ChatPromptTemplate 多消息问答】")
print(resp)

In [ ]:
# 4️⃣ ChatMessagePromptTemplate：构造消息 → LLM 调用

In [ ]:
from langchain.prompts.chat import ChatMessagePromptTemplate

# 单条 user 消息模板
user_msg_tmpl = ChatMessagePromptTemplate.from_template(
    role="user",
    template="你好，我想了解一下 {topic}"
)

# format_messages 会返回 ChatMessage 列表（这里只有一条）
msgs = user_msg_tmpl.format_messages(topic="LangChain 的 PromptTemplate 用法")

resp_msg = llm.invoke(msgs)      # 直接把消息列表丢给 llm
print("【ChatMessagePromptTemplate 提问】")
print(resp_msg.content)

# 5️⃣ System/Human/AI MessagePromptTemplate（带“历史对话”）+ LLM

In [ ]:
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
)

system_msg = SystemMessagePromptTemplate.from_template(
    "你是一个专业的运维工程师，专门排查 Linux 和 Docker 问题。"
)
human_history = HumanMessagePromptTemplate.from_template(
    "我服务器磁盘快满了，df -h 显示 overlay2 占了很多空间。"
)
ai_history = AIMessagePromptTemplate.from_template(
    "通常是 Docker 镜像层和容器层没有清理，可以先执行 docker system df 看看。"
)
current_question = HumanMessagePromptTemplate.from_template(
    "那我可以直接删除 overlay2 目录吗？"
)

history_chat_prompt = ChatPromptTemplate.from_messages([
    system_msg,
    human_history,
    ai_history,
    current_question,
])

history_chain = history_chat_prompt | llm | parser

resp = history_chain.invoke({})
print("【带历史对话的运维问答】")
print(resp)

# 6️⃣ FewShotPromptTemplate（文本 few-shot）+ LLM

In [ ]:
from langchain.prompts import PromptTemplate, FewShotPromptTemplate

examples = [
    {"text": "我很生气", "label": "负面"},
    {"text": "今天心情很好", "label": "正面"},
]

example_prompt = PromptTemplate(
    input_variables=["text", "label"],
    template="句子：{text}\n情感：{label}\n",
)

fewshot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="请判断下面句子的情感（正面/负面）：\n\n",
    suffix="句子：{input}\n情感：",
    input_variables=["input"],
)

fewshot_chain = fewshot_prompt | llm | parser

resp = fewshot_chain.invoke({"input": "这个模型效果真不错，我非常满意"})
print("【FewShotPromptTemplate 情感分类】")
print(resp)

# 7️⃣ FewShotChatMessagePromptTemplate（聊天 few-shot）+ LLM

In [ ]:
from langchain.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
)

llm = ChatOpenAI(
    base_url="https://api.siliconflow.cn/v1",
    api_key="YOUR_KEY",
    model="Qwen/Qwen2.5-32B-Instruct",
    temperature=0.3,
)

# few-shot 示例
examples = [
    {
        "question": "什么是 YOLO？",
        "answer": "YOLO 是一种实时目标检测算法，全称 You Only Look Once。"
    },
    {
        "question": "什么是向量数据库？",
        "answer": "向量数据库用于存储和检索向量，常用于相似度搜索和 RAG。"
    },
]

# 把示例展开成消息列表
example_messages = []
for ex in examples:
    example_messages.append(
        HumanMessagePromptTemplate.from_template("问：{question}").format(**ex)
    )
    example_messages.append(
        AIMessagePromptTemplate.from_template("答：{answer}").format(**ex)
    )

# 当前要问的问题
current_question_prompt = HumanMessagePromptTemplate.from_template(
    "现在请回答我的问题：{question}"
)

chat_prompt = ChatPromptTemplate.from_messages(
    example_messages + [current_question_prompt]
)

msgs = chat_prompt.format_messages(
    question="Milvus 在实际业务中有哪些典型应用场景？"
)

resp = llm.invoke(msgs)
print(resp.content)